# 🔥 Krixel Render — worker

This notebook lends its free GPU to render the studio's videos.

**Before you run (one time):**
1. Turn the GPU **ON** — Kaggle: *Settings → Accelerator → GPU T4 ×2*; Colab: *Runtime → Change runtime type → T4 GPU*.
2. Add **one Secret** (Kaggle: *Add-ons → Secrets*; Colab: 🔑 panel) — **just once, ever**:
   - `KR_GOOGLE_TOKEN` — the founder gives you this (a long block of text). Colab/Kaggle remember it, so you never paste it again.

**Then just click `Runtime → Run all`** — that ONE click runs everything (don't run cells one-by-one). Leave the tab open; it renders job after job. If it disconnects, click **Run all** again — it resumes from the last finished frame. Nothing is lost.

Full guide: `WORKER_GUIDE.md`.


In [ ]:
%%writefile render_driver.py
# -*- coding: utf-8 -*-
"""
render_driver.py  --  runs INSIDE Blender (headless), one job's frames.

    blender -b scene.blend -P render_driver.py -- \
        --engine CYCLES --outdir /work/frames --start 1 --end 1800 \
        --samples 100 --noise 0.01 --timelimit 30 --respct 100 \
        --framesfile /work/todo.txt

Design notes
------------
* Renders each frame in a single Blender session (keeps Persistent Data warm),
  but writes every frame to disk the instant it finishes -> the agent uploads it
  and the queue advances. If the session dies, finished frames are safe and the
  next run resumes (frames that already exist on disk are SKIPped).
* Prints machine-parseable lines the agent watches for:
      FPS <n>
      RANGE <start> <end>
      TOTAL <count>
      FRAMEDONE <frame> <seconds> <abspath>
      SKIP <frame>
      ALLDONE
* Honours the artist's existing scene tuning (from optimize.py); only overrides
  the job-level knobs + force-enables the GPU.
"""
import bpy, sys, os, time


def arg(flag, default=None):
    a = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []
    return a[a.index(flag) + 1] if (flag in a and a.index(flag) + 1 < len(a)) else default


scene = bpy.context.scene
engine = arg("--engine", "CYCLES")
try:
    scene.render.engine = engine
except Exception:
    engine = "CYCLES"
    scene.render.engine = engine

outdir = arg("--outdir", os.path.join(os.getcwd(), "frames"))
os.makedirs(outdir, exist_ok=True)
prefix = arg("--prefix", "lit_")

# ---- frame range ----
fs, fe = arg("--start"), arg("--end")
start = int(fs) if fs not in (None, "", "auto") else scene.frame_start
end = int(fe) if fe not in (None, "", "auto") else scene.frame_end
if end < start:
    end = start

# ---- resolution / output format ----
respct = arg("--respct")
if respct:
    scene.render.resolution_percentage = int(float(respct))
scene.render.image_settings.file_format = "PNG"
scene.render.image_settings.color_mode = "RGBA"
scene.render.use_overwrite = True
scene.render.use_file_extension = True

# ---- engine-specific knobs ----
if engine == "CYCLES":
    # Force GPU on (OPTIX preferred on RTX/T4, then CUDA/HIP/oneAPI).
    try:
        prefs = bpy.context.preferences.addons["cycles"].preferences
        picked = False
        for dt in ("OPTIX", "CUDA", "HIP", "ONEAPI"):
            try:
                prefs.compute_device_type = dt
                prefs.get_devices()
                gpus = [d for d in prefs.devices if getattr(d, "type", "CPU") != "CPU"]
                if gpus:
                    for d in prefs.devices:
                        d.use = (getattr(d, "type", "CPU") != "CPU")  # GPU only, skip slow hybrid CPU
                    picked = True
                    print("DEVICE %s (%d gpu)" % (dt, len(gpus)), flush=True)
                    break
            except Exception:
                continue
        scene.cycles.device = "GPU" if picked else "CPU"
        if not picked:
            print("DEVICE CPU (no gpu found)", flush=True)
    except Exception as e:
        print("DEVICE error: %s" % e, flush=True)

    samples = arg("--samples")
    if samples:
        scene.cycles.samples = int(samples)
    noise = arg("--noise")
    if noise:
        scene.cycles.use_adaptive_sampling = True
        scene.cycles.adaptive_threshold = float(noise)
    tl = arg("--timelimit")
    if tl is not None and tl != "":
        scene.cycles.time_limit = float(tl)          # 0 = no limit
    scene.cycles.use_denoising = True
    try:
        scene.render.use_persistent_data = True       # keep data in VRAM between frames
    except Exception:
        pass

elif engine.startswith("BLENDER_EEVEE"):
    samples = arg("--samples")
    try:
        if samples:
            scene.eevee.taa_render_samples = int(samples)
    except Exception:
        pass

# ---- which frames? explicit list (resume/gaps) or full range ----
framesfile = arg("--framesfile")
if framesfile and os.path.exists(framesfile):
    with open(framesfile) as f:
        frames = [int(x) for x in f.read().split() if x.strip()]
else:
    frames = list(range(start, end + 1))

try:
    print("FPS %s" % (scene.render.fps / max(1, scene.render.fps_base)), flush=True)
except Exception:
    print("FPS 24", flush=True)
print("RANGE %d %d" % (start, end), flush=True)
print("TOTAL %d" % (end - start + 1), flush=True)

for fr in frames:
    out = os.path.join(outdir, "%s%04d.png" % (prefix, fr))
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print("SKIP %d" % fr, flush=True)
        continue
    scene.frame_set(fr)
    scene.render.filepath = out
    t0 = time.time()
    bpy.ops.render.render(write_still=True)
    print("FRAMEDONE %d %.2f %s" % (fr, time.time() - t0, out), flush=True)

print("ALLDONE", flush=True)


In [ ]:
%%writefile agent.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Krixel Render -- cloud render agent.

Runs on a free Kaggle / Colab GPU notebook. Pulls jobs from the Google Sheet
queue, renders them with Blender (frame-by-frame, resumable), uploads every
finished frame + the final MP4 back to the user's Google Drive, and streams
live progress (frames done, sec/frame, ETA) into the Sheet so the website
dashboard can show it.

Auth: reuses the studio's existing Drive/Sheets OAuth token (the one
script_stealer.py creates) -- so the agent acts AS the user, with the user's
own Drive quota. The token JSON is passed in via the KR_GOOGLE_TOKEN env var
(stored as a private Kaggle Secret) -- never committed anywhere public.

Environment variables (set by the notebook):
    KR_GOOGLE_TOKEN   contents of oauth_token.json            (required)
    KR_SHEET_KEY      spreadsheet id of the queue sheet        (required)
    KR_BLENDER        path to the blender executable           (default: "blender")
    KR_WORKDIR        scratch dir for downloads/frames         (default: cwd)
    KR_WORKER         label shown on the dashboard             (default: auto)
    KR_OUTPUT_PARENT  Drive folder id to put outputs under      (default: My Drive root)
    KR_DRIVER         path to render_driver.py                  (default: alongside this file)

Usage:
    python agent.py            # loop: drain the queue until the session ends
    python agent.py --once     # claim + render exactly one job, then exit
"""
import os
import sys
import re
import io
import json
import time
import socket
import random
import argparse
import datetime
import subprocess

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import gspread

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/documents",
]

WORKDIR = os.environ.get("KR_WORKDIR", os.getcwd())
BLENDER = os.environ.get("KR_BLENDER", "blender")
DRIVER = os.environ.get("KR_DRIVER", os.path.join(os.path.dirname(os.path.abspath(__file__)), "render_driver.py"))
WORKER = os.environ.get("KR_WORKER") or ("%s-%s" % (socket.gethostname()[:10], random.randint(100, 999)))
OUTPUT_PARENT = os.environ.get("KR_OUTPUT_PARENT", "").strip()
OUTPUT_ROOT_NAME = "Krixel Render — Output"
STALE_MINUTES = 20            # a 'rendering' job untouched this long is assumed dead -> re-claimable
SHEET_EVERY = 12              # seconds between dashboard updates (throttle API calls)
IMG_RE = re.compile(r"(\d{4})\.png$", re.I)

os.makedirs(WORKDIR, exist_ok=True)


# ───────────────────────── auth ─────────────────────────
def get_creds():
    raw = os.environ.get("KR_GOOGLE_TOKEN")
    if raw:
        info = json.loads(raw)
    else:
        tf = os.environ.get("KR_TOKEN_FILE", "token.json")
        with open(tf) as f:
            info = json.load(f)
    creds = Credentials.from_authorized_user_info(info, SCOPES)
    if not creds.valid:
        creds.refresh(Request())
    return creds


def services():
    creds = get_creds()
    drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(os.environ["KR_SHEET_KEY"])
    return drive, sh, sh.sheet1


WORKERS_HEADER = ["worker", "state", "job", "updated"]


def heartbeat(sh, state, job=""):
    """Upsert this worker's row in the 'Workers' tab so the dashboard shows who's live."""
    try:
        try:
            w = sh.worksheet("Workers")
        except Exception:
            w = sh.add_worksheet("Workers", rows=50, cols=4)
            w.append_row(WORKERS_HEADER)
        vals = w.get_all_values()
        rownum = None
        for i, r in enumerate(vals[1:], start=2):
            if r and r[0] == WORKER:
                rownum = i
                break
        row = [WORKER, state, job, now_iso()]
        if rownum:
            w.update("A%d:D%d" % (rownum, rownum), [row])
        else:
            w.append_row(row)
    except Exception as e:
        print("heartbeat warn:", e, flush=True)


# ───────────────────────── sheet helpers ─────────────────────────
def header_map(ws):
    hdr = ws.row_values(1)
    return {name: i + 1 for i, name in enumerate(hdr)}            # name -> 1-based col


def now_iso():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat()


def is_stale(ts):
    if not ts:
        return False
    try:
        t = datetime.datetime.fromisoformat(ts)
    except Exception:
        return False
    return (datetime.datetime.utcnow() - t).total_seconds() > STALE_MINUTES * 60


def update_row(ws, cols, rownum, fields):
    """Batch-update named cells in one row with a single API call."""
    fields = dict(fields)
    fields["updated"] = now_iso()
    cells = []
    for name, val in fields.items():
        if name in cols:
            cells.append(gspread.Cell(rownum, cols[name], "" if val is None else val))
    if cells:
        ws.update_cells(cells, value_input_option="RAW")


def claim_job(ws, cols):
    """Find a queued (or stale-rendering) job and atomically claim it for this worker."""
    rows = ws.get_all_records()
    for i, row in enumerate(rows):
        rownum = i + 2
        st = (row.get("status") or "").strip()
        if st == "queued" or (st == "rendering" and is_stale(row.get("updated"))):
            update_row(ws, cols, rownum, {"status": "rendering", "worker": WORKER})
            time.sleep(1.2)                                       # let a racing worker's write land
            if ws.cell(rownum, cols["worker"]).value == WORKER:
                fresh = ws.row_values(rownum)
                return rownum, {h: (fresh[c - 1] if c - 1 < len(fresh) else "") for h, c in cols.items()}
    return None, None


# ───────────────────────── drive helpers ─────────────────────────
def drive_id(link):
    for pat in (r"/file/d/([\w-]+)", r"/folders/([\w-]+)", r"[?&]id=([\w-]+)", r"/d/([\w-]+)"):
        m = re.search(pat, link or "")
        if m:
            return m.group(1)
    return (link or "").strip()                                   # maybe they pasted a bare id


def meta(drive, fid):
    return drive.files().get(fileId=fid, fields="id,name,mimeType", supportsAllDrives=True).execute()


def download_file(drive, fid, dest):
    req = drive.files().get_media(fileId=fid)
    with io.FileIO(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk()
    return dest


def download_folder(drive, fid, destdir):
    """Download a Drive folder (flat) and return the first .blend found."""
    os.makedirs(destdir, exist_ok=True)
    blend = None
    page = None
    while True:
        res = drive.files().list(q=f"'{fid}' in parents and trashed=false",
                                 fields="nextPageToken, files(id,name,mimeType)",
                                 pageSize=500, pageToken=page).execute()
        for f in res.get("files", []):
            if f["mimeType"] == "application/vnd.google-apps.folder":
                sub = download_folder(drive, f["id"], os.path.join(destdir, f["name"]))
                blend = blend or sub
            else:
                p = os.path.join(destdir, f["name"])
                download_file(drive, f["id"], p)
                if f["name"].lower().endswith(".blend"):
                    blend = blend or p
        page = res.get("nextPageToken")
        if not page:
            break
    return blend


def find_folder(drive, name, parent):
    safe = name.replace("'", "\\'")
    q = (f"name='{safe}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
         + (f" and '{parent}' in parents" if parent else ""))
    res = drive.files().list(q=q, fields="files(id)", pageSize=1).execute()
    f = res.get("files", [])
    return f[0]["id"] if f else None


def ensure_folder(drive, name, parent=None):
    fid = find_folder(drive, name, parent)
    if fid:
        return fid
    body = {"name": name, "mimeType": "application/vnd.google-apps.folder"}
    if parent:
        body["parents"] = [parent]
    return drive.files().create(body=body, fields="id").execute()["id"]


def folder_link(fid):
    return "https://drive.google.com/drive/folders/%s" % fid


def existing_frames(drive, folder_id):
    """Frame numbers already rendered (so we resume instead of redo)."""
    done = set()
    page = None
    while True:
        res = drive.files().list(q=f"'{folder_id}' in parents and trashed=false",
                                 fields="nextPageToken, files(name)", pageSize=1000,
                                 pageToken=page).execute()
        for f in res.get("files", []):
            m = IMG_RE.search(f["name"])
            if m:
                done.add(int(m.group(1)))
        page = res.get("nextPageToken")
        if not page:
            break
    return done


def upload(drive, folder_id, path, mime, replace=True):
    name = os.path.basename(path)
    if replace:
        q = f"name='{name}' and '{folder_id}' in parents and trashed=false"
        for f in drive.files().list(q=q, fields="files(id)").execute().get("files", []):
            drive.files().delete(fileId=f["id"]).execute()
    media = MediaFileUpload(path, mimetype=mime, resumable=False)
    return drive.files().create(body={"name": name, "parents": [folder_id]},
                                media_body=media, fields="id,webViewLink").execute()


# ───────────────────────── render one job ─────────────────────────
def run_job(drive, sh, ws, cols, rownum, row):
    jid = row.get("id") or str(int(time.time()))
    name = row.get("name") or "render"
    print(f"\n=== JOB {jid} :: {name} (worker {WORKER}) ===", flush=True)

    # 1. fetch the .blend
    fid = drive_id(row.get("link"))
    m = meta(drive, fid)
    if m.get("name"):                                              # show the real file name on the dashboard right away
        name = m["name"]
        update_row(ws, cols, rownum, {"name": name})
    indir = os.path.join(WORKDIR, "in_" + jid)
    os.makedirs(indir, exist_ok=True)
    if m["mimeType"] == "application/vnd.google-apps.folder":
        blend = download_folder(drive, fid, indir)
    else:
        blend = os.path.join(indir, m["name"] if m["name"].endswith(".blend") else "scene.blend")
        download_file(drive, fid, blend)
    if not blend or not os.path.exists(blend):
        raise RuntimeError("no .blend found at that link")
    print("blend:", blend, flush=True)

    # 2. output folder in Drive (resume-aware)
    parent = OUTPUT_PARENT or ensure_folder(drive, OUTPUT_ROOT_NAME)
    out_id = ensure_folder(drive, f"{jid}__{os.path.splitext(os.path.basename(blend))[0]}", parent)
    update_row(ws, cols, rownum, {"folder": folder_link(out_id)})

    # 3. figure out the range + what's already done
    def as_int(v, d):
        try:
            return int(float(v))
        except Exception:
            return d
    start = as_int(row.get("start"), 0)
    end = as_int(row.get("end"), 0)
    if not start or not end:                                       # blank -> ask Blender for the file's range
        start, end = probe_range(blend)
    total = end - start + 1
    done_set = existing_frames(drive, out_id)
    todo = [f for f in range(start, end + 1) if f not in done_set]
    framesdir = os.path.join(WORKDIR, "fr_" + jid)
    os.makedirs(framesdir, exist_ok=True)
    todo_file = os.path.join(WORKDIR, "todo_" + jid + ".txt")
    with open(todo_file, "w") as f:
        f.write("\n".join(str(x) for x in todo))
    update_row(ws, cols, rownum, {"start": start, "end": end, "total": total, "done": len(done_set)})
    print(f"range {start}-{end}  total {total}  already {len(done_set)}  todo {len(todo)}", flush=True)

    # 4. drive Blender, parse progress, upload each frame
    cmd = [BLENDER, "-b", blend, "-P", DRIVER, "--",
           "--engine", row.get("engine") or "CYCLES", "--outdir", framesdir,
           "--start", str(start), "--end", str(end), "--framesfile", todo_file]
    for flag, key in (("--samples", "samples"), ("--noise", "noise"),
                      ("--timelimit", "timelimit"), ("--respct", "respct")):
        if str(row.get(key) or "").strip():
            cmd += [flag, str(row.get(key))]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    prior_elapsed = as_int(row.get("elapsed"), 0)
    t0 = time.time()
    done = len(done_set)
    spf_win = []
    fps = 24.0
    last_push = 0.0
    tail = []
    for line in proc.stdout:
        line = line.rstrip()
        if not line:
            continue
        tail = (tail + [line])[-25:]
        if line.startswith("FPS "):
            try:
                fps = float(line.split()[1])
            except Exception:
                pass
        elif line.startswith("FRAMEDONE"):
            parts = line.split(maxsplit=3)
            fr, secs, path = int(parts[1]), float(parts[2]), parts[3]
            try:
                upload(drive, out_id, path, "image/png", replace=False)
            except Exception as e:
                print("upload warn:", e, flush=True)
            done += 1
            spf_win = (spf_win + [secs])[-20:]
            if time.time() - last_push > SHEET_EVERY:
                spf = sum(spf_win) / len(spf_win)
                elapsed = prior_elapsed + int(time.time() - t0)
                eta = int((total - done) * spf)
                update_row(ws, cols, rownum,
                           {"done": done, "spf": round(spf, 1), "elapsed": elapsed, "eta": eta})
                heartbeat(sh, "rendering: " + (name or "")[:28], jid)
                last_push = time.time()
                print(f"  {done}/{total}  {spf:.1f}s/frame  eta {eta}s", flush=True)
        elif line.startswith(("DEVICE", "RANGE", "TOTAL", "SKIP", "ALLDONE")):
            print(" ", line, flush=True)
    rc = proc.wait()

    elapsed = prior_elapsed + int(time.time() - t0)
    done_set = existing_frames(drive, out_id)
    if len(done_set) < total:
        update_row(ws, cols, rownum, {"done": len(done_set), "elapsed": elapsed})
        raise RuntimeError("render stopped early (rc=%d): %s" % (rc, " | ".join(tail[-4:])))

    # 5. stitch -> MP4 and upload
    update_row(ws, cols, rownum, {"status": "stitching", "done": total, "elapsed": elapsed})
    mp4 = stitch(drive, out_id, framesdir, start, total, fps, name)
    if mp4:
        up = upload(drive, out_id, mp4, "video/mp4", replace=True)
        update_row(ws, cols, rownum, {"status": "done", "mp4": up.get("webViewLink"), "elapsed": elapsed})
    else:
        update_row(ws, cols, rownum, {"status": "done", "elapsed": elapsed})
    print(f"=== DONE {jid} in {elapsed}s ===", flush=True)


def probe_range(blend):
    """Headless one-liner to read the file's frame range when the job left it blank."""
    expr = ("import bpy;s=bpy.context.scene;print('KR_RANGE %d %d'%(s.frame_start,s.frame_end))")
    out = subprocess.run([BLENDER, "-b", blend, "--python-expr", expr],
                         capture_output=True, text=True).stdout
    m = re.search(r"KR_RANGE (\d+) (\d+)", out)
    return (int(m.group(1)), int(m.group(2))) if m else (1, 1)


def stitch(drive, out_id, framesdir, start, total, fps, name):
    """Make an MP4 from the frames. Downloads any frames missing locally (resumed jobs)."""
    try:
        import imageio_ffmpeg
        ff = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception as e:
        print("no ffmpeg, skipping stitch:", e, flush=True)
        return None
    # ensure every frame is local (a resumed job may have most frames only in Drive)
    have = {int(IMG_RE.search(n).group(1)) for n in os.listdir(framesdir) if IMG_RE.search(n)}
    need = [f for f in range(start, start + total) if f not in have]
    if need:
        print(f"downloading {len(need)} frames for stitch…", flush=True)
        page = None
        wanted = set(need)
        while True:
            res = drive.files().list(q=f"'{out_id}' in parents and trashed=false",
                                     fields="nextPageToken, files(id,name)", pageSize=1000,
                                     pageToken=page).execute()
            for f in res.get("files", []):
                mm = IMG_RE.search(f["name"])
                if mm and int(mm.group(1)) in wanted:
                    download_file(drive, f["id"], os.path.join(framesdir, f["name"]))
            page = res.get("nextPageToken")
            if not page:
                break
    mp4 = os.path.join(WORKDIR, re.sub(r"[^\w.-]", "_", name).rsplit(".", 1)[0] + ".mp4")
    cmd = [ff, "-y", "-framerate", str(fps or 24), "-start_number", str(start),
           "-i", os.path.join(framesdir, "lit_%04d.png"),
           "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", mp4]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("ffmpeg error:", r.stderr[-500:], flush=True)
        return None
    return mp4


# ───────────────────────── main loop ─────────────────────────
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--once", action="store_true", help="render one job then exit")
    ap.add_argument("--idle-sleep", type=int, default=20, help="seconds to wait when the queue is empty")
    args = ap.parse_args()

    drive, sh, ws = services()
    cols = header_map(ws)
    print(f"Krixel Render agent online as '{WORKER}'.  Blender: {BLENDER}", flush=True)
    heartbeat(sh, "online")

    idle = 0
    while True:
        rownum, row = claim_job(ws, cols)
        if not row:
            heartbeat(sh, "idle")
            if args.once:
                print("no jobs.", flush=True)
                return
            idle += 1
            if idle % 6 == 1:
                print("queue empty, waiting…", flush=True)
            time.sleep(args.idle_sleep)
            continue
        idle = 0
        try:
            run_job(drive, sh, ws, cols, rownum, row)
        except Exception as e:
            print("JOB FAILED:", e, flush=True)
            try:
                update_row(ws, cols, rownum, {"status": "error", "error": str(e)[:300]})
            except Exception:
                pass
        heartbeat(sh, "idle")
        if args.once:
            return


if __name__ == "__main__":
    main()


In [ ]:
# 1) Install Blender + Python deps  (run once per session)
import os, urllib.request, subprocess, sys, shutil

# Team uses Blender 5.1 — change the patch number here if your files need a different one.
BLENDER_URL = "https://download.blender.org/release/Blender5.1/blender-5.1.2-linux-x64.tar.xz"
BL_DIR = "/tmp/blender"

if not os.path.exists(os.path.join(BL_DIR, "blender")):
    os.makedirs(BL_DIR, exist_ok=True)
    print("downloading Blender…", flush=True)
    req = urllib.request.Request(BLENDER_URL, headers={"User-Agent": "Mozilla/5.0"})  # blender.org blocks non-browser UA
    with urllib.request.urlopen(req) as r, open("/tmp/blender.tar.xz", "wb") as f:
        shutil.copyfileobj(r, f)
    subprocess.run(["tar", "-xf", "/tmp/blender.tar.xz", "-C", BL_DIR, "--strip-components=1"], check=True)

BLENDER = os.path.join(BL_DIR, "blender")
print("blender:", BLENDER, "| exists:", os.path.exists(BLENDER))

# system libs Blender links against (safe if already present)
subprocess.run("apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y "
               "libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 >/dev/null 2>&1", shell=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "google-api-python-client", "google-auth", "google-auth-oauthlib",
                "gspread", "imageio-ffmpeg"], check=True)
print("deps ready ✅")


In [ ]:
# 2) Load your secrets + settings
import os

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

import json as _json
_tok = (secret("KR_GOOGLE_TOKEN") or "").strip()
assert _tok, "Add ONE secret named KR_GOOGLE_TOKEN (your Drive token). You only do this ONCE — see WORKER_GUIDE.md."
try:
    _json.loads(_tok)
except Exception:
    raise SystemExit("KR_GOOGLE_TOKEN isn't valid JSON — re-copy the ENTIRE oauth_token.json (starts with { ends with }) into the secret.")
_tokfile = os.path.abspath("kr_token.json")
with open(_tokfile, "w") as _f:
    _f.write(_tok)
os.environ["KR_TOKEN_FILE"] = _tokfile   # pass token via FILE — long env values can corrupt crossing into the !subprocess
os.environ["KR_SHEET_KEY"]  = "1EmMkEGaabbaZ7j-f6uvouh1lxy877ExjwNX17K8esQw"  # baked in — not secret

os.environ["KR_BLENDER"] = BLENDER
os.environ["KR_WORKDIR"] = "/tmp/krwork"
os.environ["KR_DRIVER"]  = os.path.abspath("render_driver.py")
def _platform():
    try:
        import kaggle_secrets  # noqa
        return "Kaggle"
    except Exception:
        pass
    try:
        import google.colab  # noqa
        return "Colab"
    except Exception:
        return "Worker"
os.environ["KR_WORKER"] = _platform() + " GPU"   # auto-detects Kaggle vs Colab
print("ready to render as worker:", os.environ["KR_WORKER"])


In [ ]:
# 3) Render!  Drains the queue until this session ends. Re-run any time — it resumes.
!python agent.py
